# Word-Level Training on 40 Videos

**Data**: 40 videos, 31K words, 11.4% positive rate
**Goal**: Train FusionMLP and verify baseline, then prepare for scale-up

In [ ]:
# Cell 1: Setup
import os, json, ast, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

# Find data
for base in ['/kaggle/input/scale221-word-level', '/kaggle/input/scale221']:
    if os.path.exists(base):
        FEAT_DIR = f'{base}/word_features'
        LABEL_DIR = f'{base}/labels'
        break
print(f'Feature dir: {FEAT_DIR}')
print(f'Label dir: {LABEL_DIR}')

In [ ]:
# Cell 2: Load all features
X_list, y_list, vids_list, ts_dict = [], [], [], {}
all_vids = []

for f in os.listdir(FEAT_DIR):
    if '_features.npy' not in f: continue
    vid = f.replace('_features.npy', '')
    feat_path = f'{FEAT_DIR}/{f}'
    label_csv = f'{LABEL_DIR}/{vid}.csv'
    if not os.path.exists(label_csv): continue
    
    X = np.load(feat_path)
    df = pd.read_csv(label_csv)
    labels = np.array([1 if str(r.get('label','O')).strip() in ('B','I','L') else 0 
                        for _, r in df.iterrows()])[:len(X)]
    ts = []
    for _, r in df.iterrows():
        try:
            ts.append(tuple(ast.literal_eval(str(r['timestamp']))))
        except: ts.append((0.0, 0.0))
    ts = np.array(ts[:len(X)])
    
    X_list.append(X); y_list.append(labels); vids_list.extend([vid]*len(X))
    ts_dict[vid] = ts; all_vids.append(vid)

X_all = np.vstack(X_list).astype(np.float32)
y_all = np.concatenate(y_list)
vids_all = np.array(vids_list)
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)

pos_rate = y_all.mean()
print(f'Videos: {len(all_vids)}, Words: {len(y_all)}, Pos rate: {100*pos_rate:.1f}%')

In [ ]:
# Cell 3: Train FusionMLP
class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

pos_weight = min((1-pos_rate)/max(pos_rate,1e-6), 3.0)
gkf = GroupKFold(n_splits=min(5,len(all_vids)))
all_probs = np.zeros(len(y_all), dtype=np.float32)
test_mask = np.zeros(len(y_all), dtype=bool)
fold_f1s = []

for fold,(tr_idx,te_idx) in enumerate(gkf.split(X_all,y_all,vids_all)):
    Xtr,Xte = X_all[tr_idx], X_all[te_idx]
    ytr,yte = y_all[tr_idx], y_all[te_idx]
    if yte.sum()==0 or ytr.sum()==0: continue
    
    scaler = StandardScaler()
    Xtr_s = np.nan_to_num(scaler.fit_transform(Xtr).astype(np.float32))
    Xte_s = np.nan_to_num(scaler.transform(Xte).astype(np.float32))
    
    model = FusionMLP(X_all.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
    Xtr_t = torch.tensor(Xtr_s); ytr_t = torch.tensor(ytr,dtype=torch.float32).unsqueeze(1)
    
    best_f1,patience,no_imp = 0,10,0
    for epoch in range(80):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0,len(Xtr_t),128):
            idx = perm[i:i+128]
            if len(idx)<2: continue
            opt.zero_grad()
            out = model(Xtr_t[idx])
            weights = torch.where(ytr_t[idx]==1, pos_weight, 1.0)
            loss = -(weights*(ytr_t[idx]*torch.log(out+1e-8)+(1-ytr_t[idx])*torch.log(1-out+1e-8))).mean()
            loss.backward(); opt.step()
        
        model.eval()
        with torch.no_grad():
            preds = (model(torch.tensor(Xte_s)).squeeze()>=0.5).int().numpy()
        f = f1_score(yte, preds, zero_division=0)
        if f>best_f1: best_f1=f; no_imp=0
        else: no_imp+=1
        if no_imp>=patience: break
    
    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(Xte_s)).squeeze().numpy()
    all_probs[te_idx]=probs; test_mask[te_idx]=True
    p=precision_score(yte,(probs>=0.5).astype(int),zero_division=0)
    r=recall_score(yte,(probs>=0.5).astype(int),zero_division=0)
    f=f1_score(yte,(probs>=0.5).astype(int),zero_division=0)
    fold_f1s.append(f)
    print(f'Fold {fold+1}: F1={f:.4f} P={p:.4f} R={r:.4f} [{epoch+1}ep]')

oof_preds = (all_probs[test_mask]>=0.5).astype(int)
oof_f1 = f1_score(y_all[test_mask], oof_preds, zero_division=0)
print(f'\nOOF Word F1@0.5: {oof_f1:.4f}')

In [ ]:
# Cell 4: IoU evaluation
def bio_to_spans(df):
    spans,i=[],0
    while i<len(df):
        lbl=str(df.iloc[i].get('label','')).strip()
        ts=ast.literal_eval(str(df.iloc[i]['timestamp']))
        if lbl=='L': spans.append((float(ts[0]),float(ts[1])))
        elif lbl=='B':
            st,en=float(ts[0]),float(ts[1]); j=i+1
            while j<len(df):
                nl=str(df.iloc[j].get('label','')).strip()
                if nl in('I','L'): en=float(ast.literal_eval(str(df.iloc[j]['timestamp']))[1]); j+=1
                else: break
            spans.append((st,en)); i=j-1
        i+=1
    return spans

def span_iou(s1,s2):
    inter=max(0.,min(s1[1],s2[1])-max(s1[0],s2[0]))
    union=max(s1[1],s2[1])-min(s1[0],s2[0])
    return inter/union if union>0 else 0.

def seg_f1(pred,gt,th=.3):
    if not pred or not gt: return 0.
    mp,mg=set(),set()
    for pi,ps in enumerate(pred):
        bi,bg=0.,-1
        for gi,gs in enumerate(gt):
            if gi in mg: continue
            iv=span_iou(ps,gs)
            if iv>=th and iv>bi: bi,bg=iv,gi
        if bg>=0: mp.add(pi); mg.add(bg)
    tp=len(mp); p=tp/len(pred) if pred else 0.; r=tp/len(gt) if gt else 0.
    return 2*p*r/(p+r) if(p+r)>0 else 0.

def merge_segs(probs,ts,thr=.5):
    spans,in_seg,start=[],False,0.
    for pr,(t0,t1) in zip(probs,ts):
        if pr>=thr and not in_seg: in_seg,start=True,t0
        elif pr<thr and in_seg: in_seg=False; spans.append((start,t0))
    if in_seg: spans.append((start,ts[-1][1]))
    return spans

for merge_th in [.5,.7,.8,.9,.95]:
    iou_ths={.1:[],.2:[],.3:[],.4:[],.5:[]}
    for vid in all_vids:
        mask=(vids_all==vid)&test_mask
        if mask.sum()==0 or vid not in ts_dict: continue
        vp=all_probs[mask]; ts=ts_dict[vid].tolist()
        if len(ts)!=len(vp): continue
        df=pd.read_csv(f'{LABEL_DIR}/{vid}.csv')
        gt=bio_to_spans(df)
        if not gt: continue
        pred=merge_segs(vp,ts,merge_th)
        for th in iou_ths: iou_ths[th].append(seg_f1(pred,gt,th))
    line=f'merge_th={merge_th}: '
    for th in [.1,.2,.3,.4,.5]:
        v=np.mean(iou_ths[th]) if iou_ths[th] else 0
        line+=f'IoU>={th}={v:.4f}  '
    print(line)